# Day 2: RAG, step 2: turn chunks into embeddings

## Step 1: Setup

In [1]:
import json
import logging
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("embeddings")

load_dotenv("../.env")
if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY is missing. Add it to the .env file in the project folder.")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-embedding-001"
DIMENSIONS = 768
CHUNKS_PATH = Path("../data/chunks.json")
EMBEDDINGS_PATH = Path("../data/chunk_embeddings.npy")
print("Connected. Model:", MODEL)

Connected. Model: gemini-embedding-001


## Step 2: Load the chunks

In [2]:
chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
print(f"{len(chunks)} chunks loaded")
pd.DataFrame(chunks)[["chunk_id", "doc_id", "page_start", "section_title", "chars"]].head()

219 chunks loaded


,chunk_id,doc_id,page_start,section_title,chars
0,heat_faq-000,heat_faq,1,Q. What is a heat wave?,471
1,heat_faq-001,heat_faq,1,Q. How to check Heatwave levels?,283
2,heat_faq-002,heat_faq,1,Q. Will I be affected by the Heatwave?,316
3,heat_faq-003,heat_faq,1,Q. What is criterion for declaring heat wave i...,627
4,heat_faq-004,heat_faq,2,Q. What is a criterion for describing Heat Wav...,214


## Step 3: Embed ONE chunk and look at it

In [3]:
one = chunks[3]
result = client.models.embed_content(
    model=MODEL,
    contents=[one["text"]],
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT", output_dimensionality=DIMENSIONS),
)
vector = np.array(result.embeddings[0].values)

print("Chunk:", one["chunk_id"], "|", one["section_title"])
print("Vector length (how many numbers):", len(vector))
print("First 8 numbers:", vector[:8].round(4))
print("Size of the arrow (norm):", round(float(np.linalg.norm(vector)), 3))

INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Chunk: heat_faq-003 | Q. What is criterion for declaring heat wave in India?
Vector length (how many numbers): 768
First 8 numbers: [ 0.0058 -0.0015  0.0072 -0.0451  0.0071  0.0019  0.0382  0.0016]
Size of the arrow (norm): 0.587


### Why we "normalise" the vectors

In [4]:
def normalise(vectors):
    vectors = np.asarray(vectors, dtype="float32")
    return vectors / np.linalg.norm(vectors, axis=-1, keepdims=True)

print("Norm after normalising:", round(float(np.linalg.norm(normalise(vector))), 3))

Norm after normalising: 1.0


## Step 4: Embed all chunks

In [5]:
BATCH_SIZE = 100
FORCE_REEMBED = False

def embed_texts(texts, task_type, retries=5):
    vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        for attempt in range(1, retries + 1):
            try:
                result = client.models.embed_content(
                    model=MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=DIMENSIONS),
                )
                vectors += [e.values for e in result.embeddings]
                break
            except Exception as error:
                if attempt == retries:
                    logger.error("Embedding failed after %d tries: %s", retries, error)
                    raise
                wait = 10 * attempt
                logger.warning("Embedding error (%s). Retry %d/%d in %ds", error, attempt, retries, wait)
                time.sleep(wait)
        logger.info("Embedded %d / %d", min(start + BATCH_SIZE, len(texts)), len(texts))
    return normalise(vectors)

if EMBEDDINGS_PATH.exists() and not FORCE_REEMBED and len(np.load(EMBEDDINGS_PATH)) == len(chunks):
    chunk_vectors = np.load(EMBEDDINGS_PATH)
    print("Loaded saved embeddings (no API calls).")
else:
    chunk_vectors = embed_texts([c["text"] for c in chunks], task_type="RETRIEVAL_DOCUMENT")
    np.save(EMBEDDINGS_PATH, chunk_vectors)
    print("Embedded and saved.")

print("Embeddings shape:", chunk_vectors.shape, "  (= chunks x numbers per vector)")

Loaded saved embeddings (no API calls).
Embeddings shape: (219, 768)   (= chunks x numbers per vector)


## Step 5: Search: question → closest chunks

In [6]:
def search(question, top_k=3):
    question_vector = embed_texts([question], task_type="RETRIEVAL_QUERY")[0]
    scores = chunk_vectors @ question_vector
    best = np.argsort(-scores)[:top_k]
    return [{**chunks[i], "score": float(scores[i])} for i in best]

def show(question, top_k=3):
    print(f"QUESTION: {question}")
    for rank, hit in enumerate(search(question, top_k), start=1):
        pages = f"p.{hit['page_start']}" + (f"-{hit['page_end']}" if hit["page_end"] != hit["page_start"] else "")
        print(f"  #{rank}  score {hit['score']:.3f}  [{hit['doc_id']}, {pages}]  {hit['section_title'][:75]}")
    print()

show("Is it dangerously hot?")

QUESTION: Is it dangerously hot?


INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


INFO Embedded 1 / 1


  #1  score 0.702  [heat_faq, p.1]  Q. How to check Heatwave levels?
  #2  score 0.696  [heat_faq, p.1]  Q. Will I be affected by the Heatwave?
  #3  score 0.694  [forecast_sop, p.171]  7.1. Objective



## Step 6: Test questions

In [7]:
for q in [
    "At what temperature is a heat wave declared for coastal stations?",
    "What counts as very heavy rainfall?",
    "What is a cyclonic storm and what wind speed does it have?",
    "How is the onset of monsoon over Kerala declared?",
    "What should I do to protect myself during a heat wave?",
    "Who won the cricket world cup?",
]:
    show(q)

QUESTION: At what temperature is a heat wave declared for coastal stations?


INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


INFO Embedded 1 / 1


  #1  score 0.818  [heat_faq, p.2]  Q. What is a criterion for describing Heat Wave for coastal stations?
  #2  score 0.763  [forecast_sop, p.175]  7.3.1. Criterion for declaring heat wave
  #3  score 0.760  [heat_faq, p.1-2]  Q. What is criterion for declaring heat wave in India?

QUESTION: What counts as very heavy rainfall?


INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


...

INFO Embedded 1 / 1


  #1  score 0.749  [heat_faq, p.13-14]  Q. What are the measures one should do to minimize the impact during heat
  #2  score 0.748  [heat_faq, p.10-11]  Q. What are the measures one should take to minimize the impact during the 
  #3  score 0.747  [forecast_sop, p.181]  7.7.1. The measures one should do to minimize the impact during heat wave

QUESTION: Who won the cricket world cup?


INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


INFO Embedded 1 / 1


  #1  score 0.515  [monsoon_faq, p.15]  41. What are the vagaries of monsoon?
  #2  score 0.508  [monsoon_faq, p.1]  1. What is the all India monthly and seasonal rainfall?
  #3  score 0.503  [monsoon_faq, p.14]  37. What is its impact on monsoon rainfall over different parts of the coun

